#1번째 시도

In [2]:
import cv2
import numpy as np
import os
from google.colab.patches import cv2_imshow # 코랩 출력용

# ==========================================
# 1. 경로 및 분석 범위 설정
# ==========================================
target_folder = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/3. weak_normalized/이름변경 후_to 0412" # 실제 경로로 수정하세요
start_idx = 0  # 분석 시작 번호
end_idx = 5    # 분석 끝 번호 (5장)

# ==========================================
# 2. 2작기용 보정 함수 (밝은 이미지 최적화)
# ==========================================
def process_stage2(img):
    # (1) LAB 색공간 변환: 밝기(L)만 따로 제어하기 위함
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # (2) CLAHE: 밝은 이미지이므로 대비 강도를 1작기(2.0)보다 낮춘 1.2~1.5 추천
    clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8, 8))
    l = clahe.apply(l)

    # (3) 다시 BGR로 결합
    img_enhanced = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

    # (4) 감마 보정: 이미지가 너무 밝다면 0.8~0.9로 설정하여 빛 번짐을 잡음
    # (1.0은 원본 유지, 1.2 이상은 더 밝게)
    gamma = 0.9
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    final_img = cv2.LUT(img_enhanced, table)

    return final_img

# ==========================================
# 3. 실행 및 결과 출력
# ==========================================
# 파일 리스트 가져오기
all_files = sorted([f for f in os.listdir(target_folder) if f.endswith(('.jpg', '.png', '.jpeg'))])
test_files = all_files[start_idx:end_idx]

print(f"총 {len(all_files)}장 중 {start_idx}번부터 {end_idx}번까지 {len(test_files)}장을 테스트합니다.")

for file_name in test_files:
    full_path = os.path.join(target_folder, file_name)

    # 한글 경로 읽기 대응
    img_array = np.fromfile(full_path, np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    if img is None:
        continue

    # 보정 실행
    result_img = process_stage2(img)

    # 비교를 위해 가로로 붙이기
    combined = np.hstack((img, result_img))

    # 출력
    print(f"\n[파일명: {file_name}] - 좌: 원본 / 우: 보정후")
    # 이미지 사이즈가 너무 크면 축소해서 출력 (보기 편하게)
    h, w = combined.shape[:2]
    cv2_imshow(cv2.resize(combined, (w//2, h//2)))

Output hidden; open in https://colab.research.google.com to view.

#1번째 보정의 적용: 30장에 대하여

In [ ]:
import cv2
import numpy as np
import os
from google.colab.patches import cv2_imshow # 코랩 출력용

# ==========================================
# 1. 경로 및 분석 범위 설정
# ==========================================
target_folder = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/3. weak_normalized" # 실제 경로로 수정하세요
save_folder="/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/5. 밝기보정/testimage"
start_idx = 0  # 분석 시작 번호
end_idx = 30    # 분석 끝 번호 (30장)

# ==========================================
# 2. 2작기용 보정 함수 (밝은 이미지 최적화)
# ==========================================
def process_stage2(img):
    # (1) LAB 색공간 변환: 밝기(L)만 따로 제어하기 위함
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # (2) CLAHE: 밝은 이미지이므로 대비 강도를 1작기(2.0)보다 낮춘 1.2~1.5 추천
    clahe = cv2.createCLAHE(clipLimit=1.0, tileGridSize=(8, 8))
    l = clahe.apply(l)

    # (3) 다시 BGR로 결합
    img_enhanced = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

    # (4) 감마 보정: 이미지가 너무 밝다면 0.8~0.9로 설정하여 빛 번짐을 잡음
    # (1.0은 원본 유지, 1.2 이상은 더 밝게)
    gamma = 0.9
    invGamma = 1.0 / gamma
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    final_img = cv2.LUT(img_enhanced, table)

    return final_img

# ==========================================
# 3. 실행 및 결과 저장
# ==========================================
# 저장 폴더 생성 (이미 존재하면 건너뜀)
os.makedirs(save_folder, exist_ok=True)

# 파일 리스트 가져오기
all_files = sorted([f for f in os.listdir(target_folder) if f.endswith(('.jpg', '.png', '.jpeg'))])
test_files = all_files[start_idx:end_idx]

print(f"총 {len(all_files)}장 중 {start_idx}번부터 {end_idx}번까지 {len(test_files)}장을 처리하고 {save_folder}에 저장합니다.")

for file_name in test_files:
    full_path = os.path.join(target_folder, file_name)
    save_path = os.path.join(save_folder, file_name) # 저장할 경로 설정

    # 한글 경로 읽기 대응
    img_array = np.fromfile(full_path, np.uint8)
    img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)

    if img is None:
        print(f"Warning: Could not read image {file_name}. Skipping.")
        continue

    # 보정 실행
    result_img = process_stage2(img)

    # 이미지 저장
    # 한글 경로 저장을 위해 cv2.imencode 사용
    is_success, im_buf_arr = cv2.imencode(".jpg", result_img)
    if is_success:
        im_buf_arr.tofile(save_path)
        print(f"[파일명: {file_name}] 보정 완료 및 저장됨.")
    else:
        print(f"Error: Failed to save image {file_name}.")

print("모든 이미지 처리가 완료되었습니다.")


총 1211장 중 0번부터 30번까지 30장을 처리하고 /content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/5. 밝기보정/testimage에 저장합니다.
[파일명: bed00_20260313_124253_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed00_20260318_113024_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed00_20260319_121935_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed00_20260403_082257_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed00_20260405_155957_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed00_20260406_103421_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed00_20260410_080135_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed01_20260308_080759_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed01_20260312_080929_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed01_20260313_082212_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed01_20260314_134123_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed01_20260315_081317_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed01_20260317_082145_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed01_20260319_080015_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed01_20260320_083634_cam2.jpg] 보정 완료 및 저장됨.
[파일명: bed01_20260322_194710_cam2.jpg] 보정 완료 

#ver2: 정규화까지 포함한 버전

In [ ]:
import cv2
import numpy as np
import os
from google.colab.patches import cv2_imshow

# ==========================================
# 1. 설정 (경로 및 범위)
# ==========================================
target_folder = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/1작기/ROI_WARP/step6_warp_images/보정 후/251213-251226/251213"
 #"/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/3. weak_normalized"
start_idx = 5
end_idx = 11

# 고정할 황금 파라미터
GOLDEN_CLIP = 1.0
GOLDEN_GAMMA = 0.9

# ==========================================
# 2. 표준화 + 보정 통합 함수
# ==========================================
def process_standardized(img):
    # --- [STEP 1] 히스토그램 스트레칭 (표준화) ---
    # 상위 1%, 하위 1%를 잘라내어 노이즈를 방지하고 0~255로 늘립니다.
    xp = [np.percentile(img, 1), np.percentile(img, 99)]
    fp = [0, 255]
    img_std = np.interp(img, xp, fp).astype('uint8')

    # --- [STEP 2] 대비 보정 (이전 코드 적용) ---
    lab = cv2.cvtColor(img_std, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=GOLDEN_CLIP, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_enhanced = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

    # --- [STEP 3] 감마 보정 (최종 톤 조절) ---
    invGamma = 1.0 / GOLDEN_GAMMA
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    final_img = cv2.LUT(img_enhanced, table)

    return final_img

# ==========================================
# 3. 실행 및 비교
# ==========================================
all_files = sorted([f for f in os.listdir(target_folder) if f.endswith(('.jpg', '.png'))])
test_files = all_files[start_idx:end_idx]

for file_name in test_files:
    full_path = os.path.join(target_folder, file_name)
    img = cv2.imdecode(np.fromfile(full_path, np.uint8), cv2.IMREAD_COLOR)

    if img is None: continue

    # 새로운 표준화 로직 적용
    result_img = process_standardized(img)

    # 출력
    combined = np.hstack((img, result_img))
    h, w = combined.shape[:2]
    print(f"\n[파일명: {file_name}] - 좌: 원본 / 우: 표준화+보정")
    cv2_imshow(cv2.resize(combined, (w//2, h//2)))

Output hidden; open in https://colab.research.google.com to view.

#2번째 버전으로 고정: 전체이미지에 대해 적용하기: 1.2작기 통합 적용 가능할듯

In [3]:
import cv2
import numpy as np
import os
import time
import concurrent.futures # Import for concurrent processing
import math # For math.ceil

# ==========================================
# 1. 경로 설정 (시즌에 상관없이 폴더만 지정)
# ==========================================
target_folder = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/3. weak_normalized/이름변경 후_to 0412"
output_folder = "/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/2. ROI_BOX/2작기/5. 밝기보정/이름변경 후_to 0412"

# 범용 황금 파라미터 (1/2작기 공용)
TRIM_PERCENT = 1    # 상하위 1% 컷팅
GOLDEN_CLIP = 1.0   # 대비 강도
GOLDEN_GAMMA = 0.9  # 2작기 빛 번짐 방지 및 1작기 톤 안정화

if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# ==========================================
# 2. 범용 보정 함수 (표준화 + 보정)
# ==========================================
def process_universal(img):
    # [1] 히스토그램 스트레칭 (표준화의 핵심)
    # 이미지의 하위/상위 1% 값을 찾아 0~255로 맵핑
    p_low, p_high = np.percentile(img, (TRIM_PERCENT, 100 - TRIM_PERCENT))
    img_std = np.clip((img - p_low) * 255.0 / (p_high - p_low), 0, 255).astype(np.uint8)

    # [2] LAB 공간 대비 보정 (CLAHE)
    lab = cv2.cvtColor(img_std, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=GOLDEN_CLIP, tileGridSize=(8, 8))
    l = clahe.apply(l)
    img_enhanced = cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2BGR)

    # [3] 감마 보정 (최종 톤 정리)
    invGamma = 1.0 / GOLDEN_GAMMA
    table = np.array([((i / 255.0) ** invGamma) * 255 for i in np.arange(0, 256)]).astype("uint8")
    final_img = cv2.LUT(img_enhanced, table)

    return final_img

# ==========================================
# 3. 개별 파일 처리 및 저장 함수 (멀티프로세싱을 위해 분리)
# ==========================================
def process_and_save_single_file(file_info):
    file_name, target_folder, output_folder = file_info
    full_path = os.path.join(target_folder, file_name)
    save_path = os.path.join(output_folder, file_name)

    # 한글 경로 대응 읽기
    img = cv2.imdecode(np.fromfile(full_path, np.uint8), cv2.IMREAD_COLOR)
    if img is None:
        return f"Warning: Could not read image {file_name}. Skipping."

    # 보정 적용
    result = process_universal(img)

    # 저장
    try:
        _, encoded_img = cv2.imencode('.jpg', result)
        encoded_img.tofile(save_path)
        return f"Processed {file_name}"
    except Exception as e:
        return f"Error saving {file_name}: {e}"

# ==========================================
# 4. 일괄 실행부 (멀티프로세싱 적용)
# ==========================================
file_list = [f for f in os.listdir(target_folder) if f.lower().endswith(('.jpg', '.png'))]
total_files = len(file_list)

print(f"총 {total_files}장의 이미지에 대해 범용 보정을 시작합니다.")
start_time = time.time()

# 4명이 동시 작업하도록 max_workers를 4로 설정
num_workers = 4

with concurrent.futures.ProcessPoolExecutor(max_workers=num_workers) as executor:
    # 각 파일에 대한 인수를 튜플로 묶어서 전달
    file_args = [(fname, target_folder, output_folder) for fname in file_list]
    futures = [executor.submit(process_and_save_single_file, arg) for arg in file_args]

    processed_count = 0
    for future in concurrent.futures.as_completed(futures):
        processed_count += 1
        res = future.result() # Get the result of the completed task
        if "Warning" in res or "Error" in res:
            print(res) # 경고 또는 에러 메시지 출력

        # 진행 상황 및 예상 완료 시간 출력
        # 10개 파일마다 또는 마지막 파일일 때 출력
        if processed_count % 10 == 0 or processed_count == total_files:
            elapsed_time = time.time() - start_time
            if processed_count > 0:
                time_per_file = elapsed_time / processed_count
                remaining_files = total_files - processed_count
                estimated_remaining_time = time_per_file * remaining_files

                # 시간을 분과 초로 변환
                elapsed_minutes = math.floor(elapsed_time / 60)
                elapsed_seconds = math.ceil(elapsed_time % 60)
                remaining_minutes = math.floor(estimated_remaining_time / 60)
                remaining_seconds = math.ceil(estimated_remaining_time % 60)

                print(f"[{processed_count}/{total_files}] 처리 중... "
                      f"(현재 소요 시간: {elapsed_minutes}분 {elapsed_seconds}초, "
                      f"예상 남은 시간: {remaining_minutes}분 {remaining_seconds}초)")

end_time = time.time()

# 최종 출력 메시지 수정
final_elapsed_time = end_time - start_time
final_elapsed_minutes = math.floor(final_elapsed_time / 60)
final_elapsed_seconds = math.ceil(final_elapsed_time % 60)

print(f"\n최종 완료! 총 {total_files}개 파일 처리. 소요시간: {final_elapsed_minutes}분 {final_elapsed_seconds}초")
print(f"모든 파일이 {output_folder}에 저장되었습니다.")


총 1255장의 이미지에 대해 범용 보정을 시작합니다.
[10/1255] 처리 중... (현재 소요 시간: 0분 3초, 예상 남은 시간: 5분 12초)
[20/1255] 처리 중... (현재 소요 시간: 0분 5초, 예상 남은 시간: 4분 19초)
[30/1255] 처리 중... (현재 소요 시간: 0분 7초, 예상 남은 시간: 4분 15초)
[40/1255] 처리 중... (현재 소요 시간: 0분 8초, 예상 남은 시간: 4분 3초)
[50/1255] 처리 중... (현재 소요 시간: 0분 10초, 예상 남은 시간: 3분 55초)
[60/1255] 처리 중... (현재 소요 시간: 0분 12초, 예상 남은 시간: 3분 50초)
[70/1255] 처리 중... (현재 소요 시간: 0분 14초, 예상 남은 시간: 3분 50초)
[80/1255] 처리 중... (현재 소요 시간: 0분 16초, 예상 남은 시간: 3분 43초)
[90/1255] 처리 중... (현재 소요 시간: 1분 8초, 예상 남은 시간: 14분 28초)
[100/1255] 처리 중... (현재 소요 시간: 1분 8초, 예상 남은 시간: 13분 4초)
[110/1255] 처리 중... (현재 소요 시간: 1분 9초, 예상 남은 시간: 11분 55초)
[120/1255] 처리 중... (현재 소요 시간: 1분 10초, 예상 남은 시간: 10분 57초)
[130/1255] 처리 중... (현재 소요 시간: 1분 10초, 예상 남은 시간: 10분 5초)
[140/1255] 처리 중... (현재 소요 시간: 1분 11초, 예상 남은 시간: 9분 20초)
[150/1255] 처리 중... (현재 소요 시간: 1분 11초, 예상 남은 시간: 8분 42초)
[160/1255] 처리 중... (현재 소요 시간: 1분 12초, 예상 남은 시간: 8분 8초)
[170/1255] 처리 중... (현재 소요 시간: 1분 12초, 예상 남은 시간: 7분 38초)
[180/1255] 처리 중... (현재 소요 시간: 1분